# Airfare insights notebook

This notebook builds summary charts from `dm.dm_route_purchase_recommendations`.

In [ ]:
import os

import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine

user = os.getenv("POSTGRES_USER", "dwh")
password = os.getenv("POSTGRES_PASSWORD", "dwh")
host = os.getenv("POSTGRES_HOST", "localhost")
port = os.getenv("POSTGRES_PORT", "5432")
db = os.getenv("POSTGRES_DB", "dwh")

engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

In [ ]:
query = """
select
    route_origin,
    route_destination,
    best_days_before_departure,
    best_search_weekday,
    best_search_hour,
    route_avg_price,
    route_price_stddev,
    route_price_cv,
    route_min_price,
    route_max_price
from dm.dm_route_purchase_recommendations
"""
df = pd.read_sql(query, engine)
df.head()

In [ ]:
route_avg = df.sort_values("route_avg_price")
fig_avg = px.bar(
    route_avg,
    x="route_origin",
    y="route_avg_price",
    color="route_destination",
    title="Average airfare by route",
)
fig_avg.show()

In [ ]:
fig_volatility = px.scatter(
    df,
    x="route_avg_price",
    y="route_price_stddev",
    color="route_destination",
    hover_data=[
        "route_origin",
        "best_days_before_departure",
        "best_search_weekday",
        "best_search_hour",
        "route_price_cv",
    ],
    title="Price level vs volatility by route",
)
fig_volatility.show()

df[
    [
        "route_origin",
        "route_destination",
        "best_days_before_departure",
        "best_search_weekday",
        "best_search_hour",
        "route_avg_price",
        "route_price_stddev",
        "route_price_cv",
    ]
].sort_values("route_avg_price").head(20)